Imports

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from skimage.transform import resize
from tifffile import imread
import torchvision.transforms as T
import torchvision.models as models
import scanpy as sc
from tifffile import TiffFile
from tqdm import tqdm
import torch

In [2]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

Load metadata and gene expression

In [3]:
BASE_PATH = r"C:\Users\mills\Desktop\Angione Lab\OC"
DAPI_FOLDER = os.path.join(BASE_PATH, "DAPI")
EMBEDDINGS_SAVE_PATH = os.path.join(BASE_PATH, "CNN_embeddings.npy")

# Z-slice files
Z_SLICE_FILES = [
    os.path.join(DAPI_FOLDER, f"HumanOvarianCancerPatient2Slice2_images_mosaic_DAPI_z{i}.tif")
    for i in range(7)
]

# ------------------------------
# Load HVG AnnData (already contains aligned metadata)
# ------------------------------
adata_hvg = sc.read_h5ad(os.path.join(BASE_PATH, "OC_adata_hvg.h5ad"))

In [4]:
# BASE_PATH = r"C:\Users\mills\Desktop\Angione Lab\OC"
# METADATA_FILE = os.path.join(BASE_PATH, "HumanOvarianCancerPatient2Slice2_cell_metadata.csv")
# GENE_FILE = os.path.join(BASE_PATH, "HumanOvarianCancerPatient2Slice2_cell_by_gene.csv")
# DAPI_FOLDER = os.path.join(BASE_PATH, "DAPI")
# EMBEDDINGS_SAVE_PATH = os.path.join(BASE_PATH, "Patient2Slice2_cell_CNN_embeddings.npy")

# # Z-slice files
# Z_SLICE_FILES = [
#     os.path.join(DAPI_FOLDER, f"HumanOvarianCancerPatient2Slice2_images_mosaic_DAPI_z{i}.tif")
#     for i in range(7)
# ]

In [5]:
# Extract cell coordinates for patch extraction
cell_coords = adata_hvg.obs[['center_x', 'center_y', 'fov']].copy()
print("Number of cells:", adata_hvg.n_obs)

Number of cells: 63072


Define patch extraction function 

In [6]:
def extract_patch_from_large_tiff(slice_path, x, y, patch_size=64):
    """
    Efficiently read a patch centered at (x, y) from a large TIFF using memmap.
    Avoids loading the full image into RAM. Handles empty patches safely.
    """
    half = patch_size // 2
    
    with TiffFile(slice_path) as tif:
        img = tif.asarray(out='memmap')
        h, w = img.shape

        # Clip coordinates
        x1 = max(int(x) - half, 0)
        y1 = max(int(y) - half, 0)
        x2 = min(int(x) + half, w)
        y2 = min(int(y) + half, h)

        # Crop the patch
        patch = img[y1:y2, x1:x2].astype(np.float32)

        # Normalize safely
        max_val = np.max(patch)
        if max_val > 0:
            patch /= max_val
        else:
            patch[:] = 0.0  # completely empty patch

        # Resize
        patch_resized = resize(patch, (patch_size, patch_size), preserve_range=True)

    return patch_resized

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Load pretrained CNN

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ResNet18 for single-channel input
resnet = models.resnet18(pretrained=True)
resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
resnet = nn.Sequential(*list(resnet.children())[:-1])  # remove classifier
resnet.to(device)
resnet.eval()

# Optional transform for standardization
transform = T.Compose([
    T.ToTensor(),
    T.Normalize(0.5, 0.5)
])

Extract embeddings for all cells (average over all z-slices)

In [ ]:
patch_size = 64
cell_embeddings = []

for idx, row in tqdm(cell_coords.iterrows(), total=cell_coords.shape[0], desc="Extracting CNN embeddings"):
    x, y, fov = row['center_x'], row['center_y'], row['fov']

    # Average patch across z-slices
    patches = [extract_patch_from_large_tiff(z_file, x, y, patch_size) for z_file in Z_SLICE_FILES]
    patch_avg = np.mean(patches, axis=0)

    patch_tensor = torch.tensor(patch_avg, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        emb = resnet(patch_tensor)
        emb = emb.view(-1).cpu().numpy()
        cell_embeddings.append(emb)

cell_embeddings = np.stack(cell_embeddings, axis=0)
print("Cell embeddings shape:", cell_embeddings.shape)

Extracting CNN embeddings:  98%|█████████▊| 61609/63072 [29:39<00:45, 32.45it/s]

Save embeddings

In [ ]:
np.save(EMBEDDINGS_SAVE_PATH, cell_embeddings)
print("Embeddings saved to:", EMBEDDINGS_SAVE_PATH)

In [ ]:
# Add embeddings as a new obsm entry
adata_hvg.obsm['X_cnn'] = cell_embeddings

adata_hvg.write(os.path.join(BASE_PATH, "adata_hvg_with_CNN_embeddings.h5ad"))
print("HVG AnnData with CNN embeddings saved.")